In [5]:
pip install pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


In [6]:
"""PS2, Part 2.3: simple logit demand in PyBLP.

This script matches the Part I logit setup exactly:
- left-hand side: ln(s_j / s_0)
- linear demand: 1 + price + iap + averagescore
- instrument: comp_apps or other_iap
- no random coefficients

The goal is to reproduce the Part I estimates in a PyBLP Problem so that the
resulting object can be used for PyBLP's equilibrium solver.
"""

from pathlib import Path
import argparse

# If these packages are missing, remove the leading # from the command below
# and run it in Terminal before running this script (it is not Python code).
# python3 -m pip install numpy pandas pyblp

import numpy as np
import pandas as pd
import pyblp


def prepare_data(path: Path) -> pd.DataFrame:
    """Build the Part I logit market-share sample."""
    data = pd.read_stata(path, convert_categoricals=False)

    for column in ['new_est', 'price', 'iap', 'averagescore']:
        data[column] = pd.to_numeric(data[column], errors='coerce')

    data['source_row'] = np.arange(1, len(data) + 1)

    data['outside'] = data['newgroup'].eq('OUTSIDE')
    data['market_size'] = data.groupby('country')['new_est'].transform('sum')
    outside_quantity = data['new_est'].where(data['outside'], 0.0)
    data['outside_share'] = outside_quantity.groupby(data['country']).transform('sum') / data['market_size']
    data['shares'] = data['new_est'] / data['market_size']

    # Match Stata: construct instruments before dropping the outside option.
    groups = data.groupby(['country', 'appgenre'], dropna=False)
    data['comp_apps'] = groups['app'].transform('size') - 1
    data['other_iap'] = (groups['iap'].transform('sum') - data['iap']) / data['comp_apps'].replace(0, np.nan)

    data['prices'] = data['price']
    data['market_ids'] = data['country']
    data['firm_ids'] = data['developer']

    inside = data.loc[~data['outside']].copy()
    if (inside['shares'] <= 0).any() or (inside['outside_share'] <= 0).any():
        raise ValueError('Logit requires positive product and outside shares.')

    inside['lhs'] = np.log(inside['shares']) - np.log(inside['outside_share'])
    return inside


def candidate_targets(sample: pd.DataFrame, acquirer: str = 'Mojang', n_targets: int = 3):
    """Pick large non-Mojang competitors in the US market as merger candidates."""
    us = sample.loc[sample['market_ids'] == 'United States'].copy()
    if us.empty:
        raise ValueError('No United States observations in the estimation sample.')

    candidate = (
        us.loc[us['firm_ids'] != acquirer]
        .groupby('firm_ids', as_index=False)['shares']
        .sum()
        .sort_values('shares', ascending=False)
        .head(n_targets)
    )
    if candidate.empty:
        raise ValueError(f'No candidate targets were found for acquirer {acquirer!r}.')

    return candidate['firm_ids'].tolist()


def estimate_logit(sample: pd.DataFrame, instrument: str, target_firms=None, acquirer: str = 'Mojang', n_targets: int = 3):
    """Reproduce the Part I IV estimate and set up the PyBLP demand problem."""
    instrument_names = ['comp_apps', 'other_iap'] if instrument == 'both' else [instrument]
    sample = sample.dropna(subset=['lhs', 'prices', 'iap', 'averagescore'] + instrument_names).copy().reset_index(drop=True)

    # PyBLP’s solver uses outside share as 1 - sum(shares) within each market.
    # To match the Part I equation exactly, keep the original lhs and normalize shares
    # so that the inside shares still sum to the same implied market share allocation.
    denominator = sample.groupby('market_ids')['shares'].transform('sum') + sample['outside_share']
    sample['original_share'] = sample['shares']
    sample['shares'] = sample['shares'] / denominator

    # Manual 2SLS check, matching the Stata Part I regression.
    x = np.column_stack([
        np.ones(len(sample)),
        sample[['prices', 'iap', 'averagescore']].to_numpy(),
    ])
    z = np.column_stack([
        np.ones(len(sample)),
        sample[['iap', 'averagescore'] + instrument_names].to_numpy(),
    ])
    projected_x = z @ np.linalg.lstsq(z, x, rcond=None)[0]
    beta_iv = np.linalg.solve(x.T @ projected_x, projected_x.T @ sample['lhs'].to_numpy())

    for idx, name in enumerate(instrument_names):
        sample[f'demand_instruments{idx}'] = sample[name]

    formulation = pyblp.Formulation('1 + prices + iap + averagescore')
    problem = pyblp.Problem(formulation, sample)
    results = problem.solve(method='1s', se_type='robust', error_behavior='raise')

    # This is the key check: the PyBLP coefficient should match the Part I 2SLS estimate.
    np.testing.assert_allclose(results.beta.ravel(), beta_iv, rtol=1e-5, atol=1e-8)
    np.testing.assert_allclose(results.delta.ravel(), sample['lhs'].to_numpy(), rtol=1e-5, atol=1e-6)

    print(f'Instrument: {instrument}')
    print('PyBLP beta:', np.asarray(results.beta).ravel())
    print('2SLS beta:', beta_iv.ravel())

    # This gives access to PyBLP’s equilibrium solver without adding random coefficients.
    market = sample.loc[sample['market_ids'] == 'United States'].copy()
    if market.empty:
        raise ValueError('No United States observations in the estimation sample.')
    costs = results.compute_costs(market_id='United States').ravel()
    equilibrium_prices = results.compute_prices(costs=costs[:, None], market_id='United States').ravel()
    print('US equilibrium prices first 5 entries:')
    print(equilibrium_prices[:5])

    # Merger counterfactual step: treat the target firm as if it were acquired by Mojang,
    # then rebuild the ownership matrix so PyBLP knows that a merged firm's products are
    # jointly priced. We do this on the U.S. market only, because the price equilibrium is
    # computed for that single-market subproblem and the ownership matrix must match its dimensions.
    if target_firms is None:
        target_firms = candidate_targets(sample, acquirer=acquirer, n_targets=n_targets)

    market = sample.loc[sample['market_ids'] == 'United States'].copy()
    # Report monetary outcomes per 1,000 market units, including the outside option.
    # This avoids treating normalized estimation shares as raw download quantities.
    baseline_shares = results.compute_shares(
        prices=equilibrium_prices[:, None], market_id='United States'
    ).ravel()
    np.testing.assert_allclose(baseline_shares, market['shares'], rtol=1e-5, atol=1e-8)
    acquirer_mask = market['firm_ids'].eq(acquirer).to_numpy()
    baseline_revenue = 1000 * equilibrium_prices * baseline_shares
    baseline_profit = 1000 * (equilibrium_prices - costs) * baseline_shares
    summary_rows = [{
        'Scenario': f'{acquirer} alone (before)',
        'Share (%)': 100 * baseline_shares[acquirer_mask].sum(),
        'Revenue / 1000': baseline_revenue[acquirer_mask].sum(),
        'Profit / 1000': baseline_profit[acquirer_mask].sum(),
    }]
    merger_gain_rows = []
    print(f'Candidate target(s): {target_firms}')
    for target in target_firms:
        post_merger = market.copy()
        # Replace the target's firm identifier with Mojang's, leaving all other firms unchanged.
        post_merger['firm_ids'] = post_merger['firm_ids'].replace(target, acquirer)
        # Ownership enters the FOCs for pricing, so this is the object that tells PyBLP the
        # merged firm's products are jointly controlled after the acquisition.
        ownership = pyblp.build_ownership(post_merger)
        counterfactual_prices = results.compute_prices(
            costs=costs[:, None],
            ownership=ownership,
            market_id='United States',
        ).ravel()
        counterfactual_shares = results.compute_shares(
            prices=counterfactual_prices[:, None], market_id='United States'
        ).ravel()
        merged_mask = market['firm_ids'].isin([acquirer, target]).to_numpy()
        merged_share = counterfactual_shares[merged_mask].sum()
        merged_revenue = 1000 * (
            counterfactual_prices[merged_mask] * counterfactual_shares[merged_mask]
        ).sum()
        merged_profit = 1000 * (
            (counterfactual_prices[merged_mask] - costs[merged_mask])
            * counterfactual_shares[merged_mask]
        ).sum()
        summary_rows.append({
            'Scenario': f'+ {target}',
            'Share (%)': 100 * merged_share,
            'Revenue / 1000': merged_revenue,
            'Profit / 1000': merged_profit,
        })
        # Compare the same portfolio before/after to isolate the merger's effect.
        merger_gain_rows.append({
            'Target': target,
            'Share change (pp)': 100 * (merged_share - baseline_shares[merged_mask].sum()),
            'Revenue change / 1000': merged_revenue - baseline_revenue[merged_mask].sum(),
            'Profit gain / 1000': merged_profit - baseline_profit[merged_mask].sum(),
        })
        # Keep original ownership labels so both merger parties are identifiable.
        price_comparison = market[['app', 'firm_ids', 'prices']].copy()
        price_comparison = price_comparison.rename(
            columns={'firm_ids': 'original_developer', 'prices': 'pre_merger_price'}
        )
        price_comparison['post_merger_price'] = counterfactual_prices
        price_comparison['price_change'] = (
            price_comparison['post_merger_price'] - price_comparison['pre_merger_price']
        )
        print(f'Post-merger prices for {target} -> {acquirer}:')
        for firm in [acquirer, target]:
            print(f'{firm} — first 5 apps (or all if fewer):')
            print(price_comparison.loc[
                price_comparison['original_developer'] == firm
            ].head(5).to_string(index=False, float_format=lambda value: f'{value:.6f}'))

    print('\nMojang outcomes: post-merger rows include all acquired apps.')
    print('Revenue and operating profit are in dataset price units per 1,000 market units.')
    print('Profit excludes fixed costs and acquisition payments; costs are held fixed.')
    print(pd.DataFrame(summary_rows).to_string(index=False, float_format=lambda value: f'{value:.4f}'))
    if merger_gain_rows:
        print('\nChanges relative to Mojang + target combined BEFORE acquisition:')
        print(pd.DataFrame(merger_gain_rows).to_string(index=False, float_format=lambda value: f'{value:.4f}'))
        best = max(merger_gain_rows, key=lambda row: row['Profit gain / 1000'])
        print(f"Largest incremental combined operating-profit gain: {best['Target']}.")
        print('This ranking excludes acquisition prices and is conditional on this demand specification.')

    return results, sample, beta_iv


# Settings
data_path = Path(
    "final_dataset_up.dta"
)
instrument = "both"  # "comp_apps", "other_iap", or "both"
target_firms = None  # Automatically select the largest competitors
n_targets = 3

# Run
inside = prepare_data(data_path)

results, sample, beta_iv = estimate_logit(
    inside,
    instrument,
    target_firms=target_firms,
    n_targets=n_targets,
)


Initializing the problem ...
Initialized the problem after 00:00:00.

Dimensions:
 T    N     F    K1    MD 
---  ----  ---  ----  ----
11   4145  950   4     5  

Formulations:
     Column Indices:         0     1      2        3      
--------------------------  ---  ------  ---  ------------
X1: Linear Characteristics   1   prices  iap  averagescore
Solving the problem ...
Estimating standard errors ...
Computed results after 00:00:00.

Problem Results Summary:
GMM     Objective    Clipped  Weighting Matrix  Covariance Matrix
Step      Value      Shares   Condition Number  Condition Number 
----  -------------  -------  ----------------  -----------------
 1    +5.716890E+00     0      +5.994471E+04      +5.121286E+04  

Cumulative Statistics:
Computation   Objective 
   Time      Evaluations
-----------  -----------
 00:00:00         1     

Beta Estimates (Robust SEs in Parentheses):
       1             prices             iap         averagescore  
---------------  --------------